## Multi-Accent and Multi-Lingual Voice Clone Demo with MeloTTS

In [1]:
import os
import torch
from openvoice import se_extractor
from openvoice.api import ToneColorConverter

C:\Users\Owner\AppData\Local\pypoetry\Cache\virtualenvs\openvoice-D7CXSivY-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



### Initialization

In this example, we will use the checkpoints from OpenVoiceV2. OpenVoiceV2 is trained with more aggressive augmentations and thus demonstrate better robustness in some cases.

In [2]:
ckpt_converter = 'checkpoints_v2/converter'
device = "cuda:0" if torch.cuda.is_available() else "cpu"
output_dir = 'outputs_v2'

tone_color_converter = ToneColorConverter(f'{ckpt_converter}/config.json', device=device)
tone_color_converter.load_ckpt(f'{ckpt_converter}/checkpoint.pth')

os.makedirs(output_dir, exist_ok=True)

C:\Users\Owner\AppData\Local\pypoetry\Cache\virtualenvs\openvoice-D7CXSivY-py3.11\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loaded checkpoint 'checkpoints_v2/converter/checkpoint.pth'
missing/unexpected keys: [] []


C:\Users\Owner\AppData\Local\pypoetry\Cache\virtualenvs\openvoice-D7CXSivY-py3.11\Lib\site-packages\wavmark\__init__.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  che

### Obtain Tone Color Embedding
We only extract the tone color embedding for the target speaker. The source tone color embeddings can be directly loaded from `checkpoints_v2/ses` folder.

In [67]:

# reference_speaker = 'resources/example_reference.mp3' # This is the voice you want to clone
reference_speaker = 'resources/owyn-reference3.mp3'
target_se, audio_name = se_extractor.get_se(reference_speaker, tone_color_converter, vad=False)

OpenVoice version: v2


In [6]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Owner\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


True

#### Use MeloTTS as Base Speakers

MeloTTS is a high-quality multi-lingual text-to-speech library by @MyShell.ai, supporting languages including English (American, British, Indian, Australian, Default), Spanish, French, Chinese, Japanese, Korean. In the following example, we will use the models in MeloTTS as the base speakers. 

In [68]:
from melo.api import TTS

texts = {
    'EN_NEWEST': "Did you ever hear a folk tale about a giant turtle?",  # The newest English base speaker model
    'EN': "Did you ever hear a folk tale about a giant turtle?",
    'ES': "El resplandor del sol acaricia las olas, pintando el cielo con una paleta deslumbrante.",
    'FR': "La lueur dorée du soleil caresse les vagues, peignant le ciel d'une palette éblouissante.",
    'ZH': "在这次vacation中，我们计划去Paris欣赏埃菲尔铁塔和卢浮宫的美景。",
    'JP': "彼は毎朝ジョギングをして体を健康に保っています。",
    'KR': "안녕하세요! 오늘은 날씨가 정말 좋네요.",
}


src_path = f'{output_dir}/tmp.wav'

# Speed is adjustable
speed = 1.0

for language, text in texts.items():
    model = TTS(language=language, device=device)
    speaker_ids = model.hps.data.spk2id
    
    for speaker_key in speaker_ids.keys():
        speaker_id = speaker_ids[speaker_key]
        speaker_key = speaker_key.lower().replace('_', '-')
        
        source_se = torch.load(f'checkpoints_v2/base_speakers/ses/{speaker_key}.pth', map_location=device)
        model.tts_to_file(text, speaker_id, src_path, speed=speed)
        save_path = f'{output_dir}/output_v2_{speaker_key}.wav'

        # Run the tone color converter
        encode_message = "@MyShell"
        tone_color_converter.convert(
            audio_src_path=src_path, 
            src_se=source_se, 
            tgt_se=target_se, 
            output_path=save_path,
            message=encode_message)

C:\Users\Owner\AppData\Local\Temp\ipykernel_65840\2504220689.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  source_se = torch.load(f'checkpoints_v2/base_speakers/ses/{

 > Text split to sentences.
Did you ever hear a folk tale about a giant turtle?
 > ===========================


100%|█| 1/1 [00:00<00:00,  5.6


 > Text split to sentences.
Did you ever hear a folk tale about a giant turtle?
 > ===========================


100%|█| 1/1 [00:00<00:00,  8.0


...

100%|█| 2/2 [00:00<00:00,  8.1


 > Text split to sentences.
彼は毎朝ジョギングをして体を健康に保っています.
 > ===========================


100%|█| 1/1 [00:00<00:00,  7.4


 > Text split to sentences.
안녕하세요! 오늘은 날씨가 정말 좋네요.
 > ===========================


100%|█| 1/1 [00:00<00:00,  3.8


In [69]:
import IPython
import os

files = os.listdir("outputs_v2/")

for file in files:
    path = f"outputs_v2/{file}"
    print(path)
    IPython.display.display(IPython.display.Audio(path))

outputs_v2/output_v2_en-au.wav


outputs_v2/output_v2_en-br.wav


outputs_v2/output_v2_en-default.wav


outputs_v2/output_v2_en-india.wav


outputs_v2/output_v2_en-newest.wav


...

outputs_v2/output_v2_fr.wav


outputs_v2/output_v2_jp.wav


outputs_v2/output_v2_kr.wav


outputs_v2/output_v2_zh.wav


outputs_v2/tmp.wav


In [70]:
from melo.api import TTS

language="EN"

model = TTS(language=language, device=device)

speaker_ids = model.hps.data.spk2id
print(speaker_ids)

speaker_id = speaker_ids["EN-US"]
speaker_key = speaker_key.lower().replace('_', '-')

src_path = f'{output_dir}/tmp.wav'
# Speed is adjustable
speed = 0.9

text = "I love Charlie, Frankie and Mama.  I also love Bobes and Chew... They are the buddiest"

source_se = torch.load(f'checkpoints_v2/base_speakers/ses/{speaker_key}.pth', map_location=device)
model.tts_to_file(text, speaker_id, src_path, speed=speed)
save_path = f'{output_dir}/output_v2_{speaker_key}.wav'

# Run the tone color converter
encode_message = "@MyShell"
tone_color_converter.convert(
    audio_src_path=src_path, 
    src_se=source_se, 
    tgt_se=target_se, 
    output_path=save_path,
    message=encode_message)

IPython.display.display(IPython.display.Audio(save_path))
IPython.display.display(IPython.display.Audio(f'{output_dir}/tmp.wav'))

C:\Users\Owner\AppData\Local\Temp\ipykernel_65840\1738112036.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  source_se = torch.load(f'checkpoints_v2/base_speakers/ses/{

{'EN-US': 0, 'EN-BR': 1, 'EN_INDIA': 2, 'EN-AU': 3, 'EN-Default': 4}
 > Text split to sentences.
I love Charlie, Frankie and Mama. I also love Bobes and Chew. . . They are the buddiest
 > ===========================


100%|█| 1/1 [00:00<00:00,  3.4


In [74]:
from faster_whisper import WhisperModel, BatchedInferencePipeline

model_size = "large-v3"

# Run on GPU with FP16
model = WhisperModel(model_size, device="cuda", compute_type="int8")
batched_model = BatchedInferencePipeline(model=model)

segments, info = batched_model.transcribe("resources/owyn-reference.mp3", beam_size=5, batch_size=8)

print("Detected language '%s' with probability %f" % (info.language, info.language_probability))

for segment in segments:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))


Detected language 'en' with probability 1.000000
[0.00s -> 24.99s]  I am a bunny by Richard Scarry. I am a bunny. My name is Nicholas. I live in a hollow tree. In the spring, I like to pick flowers. I chase butterflies and the butterflies chase me. In the summer, I like to lie in the sun and watch the birds. And I like to watch the frogs in the pond. When it rains, I keep dry under a toadstool. I blow the dandelion seeds into the air.
[24.99s -> 38.60s]  In the fall, I like to watch the leaves falling from the trees. I watch the animals getting ready for winter. And when winter comes, I watch the snow falling from the sky. Then I curl up in my hollow tree and dream about spring.


In [75]:
segments2, info2 = batched_model.transcribe("resources/example_reference.mp3", beam_size=5, batch_size=8);
print("Detected language '%s' with probability %f" % (info2.language, info2.language_probability))

for segment in segments2:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))


Detected language 'en' with probability 1.000000
[0.00s -> 26.19s]  When I was a wanted man, the resistance gave me a lot of help. The time I spent with Guru was brief, but he left a deep impression on me. He's the type of person who says whatever's on his mind. He shares the highs and lows of his subordinates, and is never afraid to draw his sword for the sake of a friend. That's the kind of person I can really get along with. She defeated my friend in a duel before the throne, which I accept as proof of her great strength. But she uses that strength to serve as an oppressor's lackey.
[26.19s -> 48.40s]  leading the Kuju clan's troops to relentlessly seek out and confiscate visions. This, I cannot forgive. I've asked myself this question many times since leaving Inazuma. Do I simply resent the Raiden Shogun because of what happened in that duel? Because of the lethal stroke she dealt, my dear friend. I've thought about this a good long time. And I believe the answer is no. My friend d

In [76]:
segments3, info3 = batched_model.transcribe("resources/owyn-reference3.mp3", beam_size=5, batch_size=8)

print("Detected language '%s' with probability %f" % (info3.language, info3.language_probability))

for segment in segments3:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))


Detected language 'en' with probability 1.000000
[0.00s -> 24.98s]  I am a bunny by Richard Scarry. I am a bunny. My name is Nicholas. I live in a hollow tree. In the spring, I like to pick flowers. I chase butterflies and the butterflies chase me. In the summer, I like to lie in the sun and watch the birds. And I like to watch the frogs in the pond. When it rains, I keep dry under a toadstool. I blow the dandelion seeds into the air.
[24.98s -> 50.22s]  In the fall, I like to watch the leaves falling from the trees. I watch the animals getting ready for winter. And when winter comes, I watch the snow falling from the sky. Then I curl up in my hollow tree and dream about spring. Soy un conejito. Soy un conejito. Mi nombre es Nicolas. Vivo en un árbol hueco. En la primavera, me gusta recoger flores.
[50.22s -> 80.13s]  Persigo a las mariposas y las mariposas me persiguen a mí. En el verano me gusta tumbarme al sol y mirar a los pájaros. Y me gusta ver a las rañas en la charca. Cuando ll